# Homework: Sentiment Steering and Sparse Autoencoders

This assignment explores two mechanistic interpretability techniques:

1. Sentiment steering via activation addition. 
2. Sparse autoencoder on model activations. 

In [1]:
!pip install -q torch matplotlib transformer_lens transformers datasets 

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.3/945.3 kB 17.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.7 MB/s eta 0:00:00


In [3]:
import torch
from transformer_lens import HookedTransformer
import numpy as np

model = HookedTransformer.from_pretrained('gpt2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

## Part 1 – Sentiment Steering via residual (3 points)

Your task is to steer model into good/bad generations and see the results.

In [4]:
from typing import List

# This is enough to steer! You may experiment with dataset as you want
positive_sentences = [
    'I love this product, it works wonderfully!',
    'This is the best day I have ever had.',
    'I am feeling fantastic and everything is great.',
    'What a delightful surprise!',
    'The food was amazing and the service was excellent.'
]

negative_sentences = [
    'I hate this product, it is terrible.',
    'This is the worst day of my life.',
    'I am feeling awful and everything is bad.',
    'What a horrible experience.',
    'The food was disgusting and the service was terrible.'
]

# Function to collect average residual activations for a list of sentences
def collect_average_residuals(sent_list: List[str]):
    # Hint: You need *residual pre hook* from each layer
    avgs = [None] * 12
    for i in range(len(sent_list)):
        _, cache = model.run_with_cache(sent_list[i])
        for l in range(12):
            if i == 0:
                avgs[l] = cache[f'blocks.{l}.hook_resid_pre'].mean(dim=1).squeeze()
            else:
                avgs[l] = avgs[l] + cache[f'blocks.{l}.hook_resid_pre'].mean(dim=1).squeeze()
    
    return [x / len(sent_list) for x in avgs]

pos_avgs = collect_average_residuals(positive_sentences)
neg_avgs = collect_average_residuals(negative_sentences)

steering_vectors = [pos - neg for pos, neg in zip(pos_avgs, neg_avgs)]

In [5]:
# Function to generate text with steering applied
def generate_with_steering(prompt, max_new_tokens=20, coef=0.0):
    tokens = model.to_tokens(prompt).to(device)

    for _ in range(max_new_tokens):
        # Define hooks that add the steering vector times coef at every layer
        hooks = []
        # Hint: make hooks that add steering vecor with coef to each layer
        def make_hook(layer_idx):
            def hook_fn(resid, hook):
                vec = steering_vectors[layer_idx].to(resid.device)
                return resid + coef * vec.view(1, 1, -1)
            return hook_fn

        for layer in range(model.cfg.n_layers):
            hooks.append((f'blocks.{layer}.hook_resid_pre', make_hook(layer)))

        logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        next_token = logits[0, -1].argmax().unsqueeze(0)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

        if next_token == model.tokenizer.eos_token_id:
            break

    return model.to_string(tokens[0, 1:])

prompt = 'The movie that I watched yesterday was'
print('Neutral completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=0.0))
print('Positive-steered completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=0.3))
print('Negative-steered completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=-0.3))

Neutral completion:
The movie that I watched yesterday was a bit of a disappointment. I was hoping for a more mature, more mature, more mature movie
Positive-steered completion:
The movie that I watched yesterday was a great one. I was able to watch it with my wife and her family. I was able
Negative-steered completion:
The movie that I watched yesterday was a horrible movie. It was a horrible movie. It was a horrible movie. It was a horrible


You should see how this is much more effective than tinkering with attantion heads from seminar.

## Part 2 – Sparse Autoencoder on Residual Activations (4 + bonus)

This is compute intensive part and you may adjust hyperparameters to your liking. Try to get meaningful results but in the end it might be compute bound. You still can get max points.


In [6]:
from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
texts = ds["text"]

texts = [t for t in texts if len(t.strip()) > 0]
dataset_sentences = texts[:] # You may want to adjust heres

print("Total lines:", len(dataset_sentences))

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Total lines: 23767


In [7]:
from tqdm import tqdm 

last_layer = model.cfg.n_layers - 1

activations = []
token_to_sentence = []
with torch.no_grad():
    for sent_id, sent in tqdm(enumerate(dataset_sentences)):
        tokens = model.to_tokens(sent).to(device)
        _, cache = model.run_with_cache(tokens)
        # Take the *residual stream* at the last layer
        resid = cache[f'blocks.{last_layer}.hook_resid_post']
        activations.append(resid.squeeze(0))
        token_to_sentence.extend([sent_id] * resid.shape[1]) # store sent_id per activated token

    activations = torch.cat(activations, dim=0)
    token_to_sentence = torch.tensor(token_to_sentence)
    print('Activation dataset shape:', activations.shape)

23767it [14:27, 27.41it/s]


Activation dataset shape: torch.Size([2415651, 768])


#### Your task is to use any SAE try to disentangle features from residual. In the end you will look at top sentences that activate certain features.

Classic approach is enc + relu + dec \
For loss: mse + coef * l1 on hiddden \
But feel free to experiment!

In [8]:
import torch.nn as nn

class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.enc = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dec = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        hid = self.relu(self.enc(x))
        restored = self.dec(hid)
        return hid, restored

In [9]:
import torch.optim as optim

input_dim = activations.shape[1]
hidden_dim = input_dim * 4
learning_rate = 0.0001
num_epochs = 100

sae = SAE(input_dim, hidden_dim).to(device).train()
optimizer = optim.Adam(sae.parameters(), lr=learning_rate)
mse = nn.MSELoss()

dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(activations), batch_size=int(2**18), shuffle=True, drop_last=True)

for epoch in tqdm(range(num_epochs)):
    epoch_loss, epoch_recon, epoch_l1 = 0, 0, 0
    for (batch, ) in tqdm(dl):
        optimizer.zero_grad()

        hid, restored = sae(batch)
        recon_loss = mse(restored, batch)
        l1_loss = hid.abs().mean()
        
        loss = recon_loss + 0.005 * l1_loss
        
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_recon += recon_loss.item()
        epoch_l1 += l1_loss.item()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(dl):.6f}, Recon: {epoch_recon / len(dl):.6f}, L1: {epoch_l1 / len(dl):.6f}')

  5%|▌         | 5/100 [01:42<32:31, 20.54s/it]

Epoch 5/100, Loss: 52.271467, Recon: 52.240580, L1: 6.177572


 10%|█         | 10/100 [03:26<31:02, 20.70s/it]

Epoch 10/100, Loss: 23.704759, Recon: 23.667314, L1: 7.489023


 15%|█▌        | 15/100 [05:11<29:37, 20.91s/it]

...

Epoch 90/100, Loss: 0.479635, Recon: 0.416324, L1: 12.662354


 95%|█████████▌| 95/100 [32:41<01:42, 20.60s/it]

Epoch 95/100, Loss: 0.420044, Recon: 0.356391, L1: 12.730533


100%|██████████| 100/100 [34:23<00:00, 20.63s/it]

Epoch 100/100, Loss: 0.374441, Recon: 0.310493, L1: 12.789533


Below is some code that prints tokens that activate certain neuron the most. You are free to change it!

In [16]:
sae.eval()
with torch.no_grad():
    _, h = sae(activations[:1000000])
    h = h.cpu().numpy()

# For selected neuron indices, find the sentences with highest activation
selected_neurons = [0, 1, 2]  # you can choose other indices


for neuron in selected_neurons:
    activations_neuron = h[:, neuron]
    top_idx = activations_neuron.argsort()[::-1][:5]
    print(f'Neuron {neuron}:')
    for rank, idx in enumerate(top_idx, start=1):
        sent_id = token_to_sentence[idx].item()
        text = dataset_sentences[sent_id]

        toks = model.to_tokens(text)[0, 1:]
        str_toks = model.to_str_tokens(toks)

        token_positions = (token_to_sentence == sent_id).nonzero(as_tuple=True)[0]
        local_pos = (token_positions == idx).nonzero(as_tuple=True)[0].item()
        cropped = "".join(str_toks[:local_pos + 1])

        print(f"Top {rank}: global token {idx}, sentence {sent_id}, local token {local_pos}")
        print(cropped)
        print()

Neuron 0:
Top 1: global token 913231, sentence 9070, local token 45
 Reala ( リアラ , Riara ) is the main female protagonist , appearing suddenly and holding an air of mystery . While she bears an overly @-@ strong sense of responsibility , she is also bright and highly inqu

Top 2: global token 122692, sentence 1182, local token 48
 Although Jordan has done much to increase the status of the game , some of his impact on the game 's popularity in America appears to be fleeting . Television ratings in particular increased only during his time in the league , and Finals ratings have not returned

Top 3: global token 161997, sentence 1493, local token 318
 Despite its modest debut week , Wrapped in Red began to gain traction at the beginning of the holiday season , selling up to 131 @,@ 000 copies during the Thanksgiving week . It experienced its best sales week after benefiting from NBC 's premiere broadcast of Cautionary Christmas Music Tale , selling up to 136 @,@ 000 copies on its sevent